## Hand Gesture Detection *for Mouse Control*

An application that uses hand gestures to control a virtual
mouse cursor. By recognizing specific hand gestures (e.g., pointing, open hand, fist), the
application allows users to control mouse movements and clicks via hand gestures detected in a webcam feed. This project combines computer vision for gesture recognition with
interactive elements to simulate mouse control.

**Team Members:**

- Mira Emad - 202200319
- Alya Mohamed - 202202900
- Youssef Fathy - 202202193
- Habiba Khalil - 202200720

## DISCLAIMER

If you are running this script on Google Colab, follow the following instructions:


1. Install dependencies on your local machine (not in Colab):
   Open PowerShell / Terminal and run:
       **pip install mediapipe opencv-python pyautogui numpy**

       OR
       
       **pip install -r requirements.txt**

2. Start a local Jupyter server:
       **jupyter server**
   - This will display a URL, e.g.:
       http://localhost:8888/?token=whjdvhadvjskadjsdvjkvk
   - Copy this full URL.

3. Connect Google Colab to the local runtime:
   - Open your Colab notebook.
   - Click the arrow next to "Connect" in the top right → select **"Connect to local runtime"**.
   - Paste the URL from step 2 and click "Connect".

4. Update the hand landmarker path in the script if needed:
       **folder_path** = r"C:\Users\YourName\Desktop\

5. Run the notebook:
   - Your code will now execute on your **local machine**, with access to:
       - Webcam (for hand gesture detection)
       - Mouse (PyAutoGUI control)

*For more information, go to:*

https://medium.com/@missoctober959/how-to-guide-streaming-video-inputs-in-google-colab-using-opencv-c5c9126e9103


## Code

In [ ]:
import urllib.request
import os

folder_path = r"C:\Users\mirae\Desktop\HandGestureDetection"

model_url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
model_path = os.path.join(folder_path, "hand_landmarker.task")

urllib.request.urlretrieve(model_url, model_path)

('C:\\Users\\mirae\\Desktop\\HandGestureDetection\\hand_landmarker.task',
 <http.client.HTTPMessage at 0x2b4a3222910>)

In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

import numpy as np
import pyautogui

In [ ]:
def is_finger_up(hand_landmarks, tip_id, pip_id):
    """
    Check if a finger is pointing up or folded down.

    - Gets the tip position (top of finger)
    - Gets the pip position (middle joint)
    - If tip is ABOVE pip (tip has smaller y than pip) then the finger is up

    tip_id: The landmark number for finger tip
    pip_id: The landmark number for middle joint

    Returns: True if finger is up, False if down
    """
    tip = hand_landmarks[tip_id]
    pip = hand_landmarks[pip_id]

    return tip.y < pip.y

def is_finger_pointing(hand_landmarks):
    """
    Check if the hand is making a pointing gesture,
    regardless of direction.
    """
    # Index finger landmarks
    index_tip = hand_landmarks[8]
    index_pip = hand_landmarks[6]
    index_base = hand_landmarks[5]

    # Other fingers: make sure they are folded
    other_fingers = [
        is_finger_up(hand_landmarks, 12, 10), # middle
        is_finger_up(hand_landmarks, 16, 14), # ring
        is_finger_up(hand_landmarks, 20, 18) # pinky
    ]

    # Index finger must be extended (tip far from base)
    dx = index_tip.x - index_base.x
    dy = index_tip.y - index_base.y
    index_length = (dx**2 + dy**2) ** 0.5

    INDEX_EXTENSION_THRESHOLD = 0.08

    return (
        index_length > INDEX_EXTENSION_THRESHOLD and
        not any(other_fingers)
    )

def is_thumb_up(hand_landmarks):
    """
    Check if thumb is extended.

    Thumb moves sideways, not up/down.
    So we check x-coordinate instead of y-coordinate.

    Returns: True if thumb is extended, False if folded
    """
    thumb_tip = hand_landmarks[4]   # Thumb tip
    thumb_ip = hand_landmarks[3]    # Thumb joint

    pinky_tip = hand_landmarks[20] # Pinky tip

    # Check wether this is the right or left hand,
    # if right thumb extended, tip x < ip x
    # if left thumb extended, tip x > ip x
    if pinky_tip.x > thumb_tip.x:
        return thumb_tip.x < thumb_ip.x
    else:
        return thumb_tip.x > thumb_ip.x


def get_all_fingers_up(hand_landmarks):
    """
    Check which fingers are up.

    Returns a list of 5 True/False values:
    [thumb, index, middle, ring, pinky]

    Example: [False, True, False, False, False] means only index finger is up
    """
    fingers = []

    # Thumb
    fingers.append(is_thumb_up(hand_landmarks))

    # Index finger, tip=8, pip=6
    fingers.append(is_finger_up(hand_landmarks, 8, 6))

    # Middle finger, tip=12, pip=10
    fingers.append(is_finger_up(hand_landmarks, 12, 10))

    # Ring finger, tip=16, pip=14
    fingers.append(is_finger_up(hand_landmarks, 16, 14))

    # Pinky finger, tip=20, pip=18
    fingers.append(is_finger_up(hand_landmarks, 20, 18))

    return fingers


def recognize_gesture(hand_landmarks, frame_width, frame_height):
    """
    Figure out what gesture the hand is making.

    Uses finger states for static gestures (FIST, OPEN, etc.)
    and get_direction() for pointing direction.

    Returns: A string like
    "POINTING_UP", "FIST", "PEACE", "OPEN", etc.
    """
    fingers = get_all_fingers_up(hand_landmarks)

    # Index pointing
    if is_finger_pointing(hand_landmarks):
        direction = get_direction(hand_landmarks, frame_width, frame_height)
        return f"POINTING_{direction}"

    # Fist
    if fingers == [False, False, False, False, False]:
        return "FIST"

    # Peace sign
    if fingers == [False, True, True, False, False]:
        return "PEACE"

    # Open hand
    if fingers == [True, True, True, True, True]:
        return "OPEN"

    return "UNKNOWN"

def get_direction(hand_landmarks, frame_width, frame_height):
    """
    Figure out which direction the index finger is pointing.

    Compares finger tip to finger base position.

    Returns: "UP", "DOWN", "LEFT", "RIGHT", "UP_LEFT", etc.
    """
    # Get index finger tip position
    index_tip = hand_landmarks[8]

    # Get index finger base/
    index_base = hand_landmarks[5]

    direction_x = index_tip.x - index_base.x
    direction_y = index_tip.y - index_base.y

    # Threshold - how much movement to count as a direction
    # Prevents detecting tiny angles as directions
    threshold = 0.05  # 5% of frame size

    # Check horizontal (left/right)
    horizontal = ""
    if direction_x < -threshold:
        horizontal = "LEFT"
    elif direction_x > threshold:
        horizontal = "RIGHT"

    # Check vertical (up/down)
    vertical = ""
    if direction_y < -threshold:
        vertical = "UP"
    elif direction_y > threshold:
        vertical = "DOWN"

    if vertical and horizontal:
        return f"{vertical}_{horizontal}" # Like "UP_LEFT"
    elif vertical:
        return vertical # Like "UP"
    elif horizontal:
        return horizontal # Like "LEFT"
    else:
        return "CENTER" # Near the center

In [ ]:
SCREEN_WIDTH, SCREEN_HEIGHT = pyautogui.size()
pyautogui.FAILSAFE = True
pyautogui.PAUSE = 0.01

# Click debouncing
frames_since_left_click = 999
frames_since_right_click = 999
click_cooldown = 15

def can_left_click():
    global frames_since_left_click, click_cooldown
    frames_since_left_click += 1
    if frames_since_left_click >= click_cooldown:
        frames_since_left_click = 0
        return True
    return False

def can_right_click():
    global frames_since_right_click, click_cooldown
    frames_since_right_click += 1
    if frames_since_right_click >= click_cooldown:
        frames_since_right_click = 0
        return True
    return False

def move_mouse(x, y):
    # Clamp to screen bounds
    x = max(0, min(SCREEN_WIDTH - 1, x))
    y = max(0, min(SCREEN_HEIGHT - 1, y))
    pyautogui.moveTo(x, y)

def left_click():
    pyautogui.click(button='left')

def right_click():
    pyautogui.click(button='right')

In [ ]:
# Setup MediaPipe
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)
detector = vision.HandLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)

print("Point in a direction to move cursor that way")
print("FIST = Left click | PEACE = Right click | OPEN HAND = Stop")

frame_number = 0

# Movement speed (pixels per frame)
MOVE_SPEED = 8

while True:
    success, frame = cap.read()
    if not success:
        break

    frame = cv2.flip(frame, 1)
    height, width, _ = frame.shape

    # Detect hand
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    timestamp = int(frame_number * 1000 / 30)
    result = detector.detect_for_video(mp_image, timestamp)
    frame_number += 1

    if result.hand_landmarks:
        hand = result.hand_landmarks[0]

        gesture = recognize_gesture(hand, width, height)

        if gesture.startswith("POINTING"):
            # Get current cursor position
            current_x, current_y = pyautogui.position()

            # Move cursor based on DIRECTION
            if gesture == "POINTING_UP":
                move_mouse(current_x, current_y - MOVE_SPEED)

            elif gesture == "POINTING_DOWN":
                move_mouse(current_x, current_y + MOVE_SPEED)

            elif gesture == "POINTING_LEFT":
                move_mouse(current_x - MOVE_SPEED, current_y)

            elif gesture == "POINTING_RIGHT":
                move_mouse(current_x + MOVE_SPEED, current_y)

            elif gesture == "POINTING_UP_LEFT":
                move_mouse(current_x - MOVE_SPEED, current_y - MOVE_SPEED)

            elif gesture == "POINTING_UP_RIGHT":
                move_mouse(current_x + MOVE_SPEED, current_y - MOVE_SPEED)

            elif gesture == "POINTING_DOWN_LEFT":
                move_mouse(current_x - MOVE_SPEED, current_y + MOVE_SPEED)

            elif gesture == "POINTING_DOWN_RIGHT":
                move_mouse(current_x + MOVE_SPEED, current_y + MOVE_SPEED)

            elif gesture == "POINTING_CENTER":
                # Don't move if pointing at camera
                pass

        elif gesture == "FIST":
            if can_left_click():
                left_click()
                print("LEFT CLICK")

        elif gesture == "PEACE":
            if can_right_click():
                right_click()
                print("RIGHT CLICK")

        # Draw hand points
        for i, landmark in enumerate(hand):
            x = int(landmark.x * width)
            y = int(landmark.y * height)
            color = (0, 0, 255) if i == 8 else (0, 255, 255)
            size = 10 if i == 8 else 4
            cv2.circle(frame, (x, y), size, color, -1)

        # Show gesture
        cv2.putText(frame, f"Gesture: {gesture}", (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    else:
        cv2.putText(frame, "No hand detected", (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.imshow('Virtual Mouse - Direction Control', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
detector.close()

Point in a direction to move cursor that way
FIST = Left click | PEACE = Right click | OPEN HAND = Stop
RIGHT CLICK
RIGHT CLICK
RIGHT CLICK
RIGHT CLICK
RIGHT CLICK
LEFT CLICK
LEFT CLICK
LEFT CLICK
LEFT CLICK
LEFT CLICK
LEFT CLICK
RIGHT CLICK
RIGHT CLICK
RIGHT CLICK
RIGHT CLICK
RIGHT CLICK
RIGHT CLICK
LEFT CLICK
LEFT CLICK
LEFT CLICK
LEFT CLICK
LEFT CLICK
LEFT CLICK
LEFT CLICK
LEFT CLICK
RIGHT CLICK
RIGHT CLICK
RIGHT CLICK
RIGHT CLICK
LEFT CLICK
LEFT CLICK
LEFT CLICK
RIGHT CLICK
RIGHT CLICK
RIGHT CLICK
LEFT CLICK
LEFT CLICK
RIGHT CLICK
RIGHT CLICK
RIGHT CLICK
RIGHT CLICK
RIGHT CLICK
RIGHT CLICK
RIGHT CLICK


FailSafeException: PyAutoGUI fail-safe triggered from mouse moving to a corner of the screen. To disable this fail-safe, set pyautogui.FAILSAFE to False. DISABLING FAIL-SAFE IS NOT RECOMMENDED.